# Lesson 02 Lab — Blocked Programs versus CUDA Threads

**Puzzle:** When program_id, blocked tensors, and scalar threads change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates program_id, blocked tensors, and scalar threads and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

CUDA exposes scalar threads grouped into blocks. Triton exposes a grid of program instances; each instance evaluates tensor expressions such as tl.arange. The mapping can resemble one CUDA block, but it is a reasoning aid rather than an identity.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["program_id, blocked tensors, and scalar threads"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

Treating each lane of a blocked tensor as an independently scheduled CUDA thread leads to wrong assumptions about synchronization and storage.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 2
LESSON_TITLE = 'Blocked Programs versus CUDA Threads'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260815
}


## 5. Freeze the experiment

**Experiment:** Implement vector addition with 256 elements per Triton program and compare it with torch.add.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 0.02113599982112646,
  "secondary": 0.013584000058472157,
  "max_abs_error": 0.0,
  "passed": true,
  "details": {
    "programs": 16384,
    "block_elements": 256,
    "triton_samples_ms": [
      0.035071998834609985,
      0.02675200067460537,
      0.023072000592947006,
      0.02319999970495701,
      0.020191999152302742,
      0.0225600004196167,
      0.02208000048995018,
      0.02131200022995472,
      0.020959999412298203,
      0.021344000473618507,
      0.02051199972629547,
      0.020287999883294106,
      0.021568000316619873,
      0.019039999693632126,
      0.0208320003002882,
      0.019487999379634857,
      0.02051199972629547,
      0.02131200022995472,
      0.020255999639630318,
      0.020576000213623047
    ],
    "pytorch_samples_ms": [
      0.013311999849975109,
      0.013055999763309956,
      0.013567999936640263,
      0.013791999779641628,
      0.01360000018030405,
      0.01360000018030405,
      0.0144640002399683,
      0.0137280002

## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Triton median | 0.0211 ms |
| torch.add median | 0.0136 ms |
| Maximum absolute error | 0.000e+00 |
| Acceptance gate | true |


## 8. Explain without overclaiming

16,384 blocked programs covered 4,194,304 scalar elements; correctness matched torch.add and the two timings are reported separately.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Reason in blocked programs first; descend to warp and instruction details only when generated code or counters require it.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 2,
  "title": "Blocked Programs versus CUDA Threads",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260815
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 0.02113599982112646,
    "secondary": 0.013584000058472157,
    "max_abs_error": 0.0,
    "passed": true,
    "details": {
      "programs": 16384,
      "block_elements": 256,
      "triton_samples_ms": [
        0.035071998834609985,
        0.02675200067460537,
        0.023072000592947006,
        0.02319999970495701,
        0.020191999152302742,
        0.0225600004196167,
        0.02208000048995018,
        0.02131200022995472,
        0.020959999412298203,
        0.021344000473618507,
        0.02051199972629547,
        0.020287999883294106,
        0.

## 10. Make the bounded decision

> Reason in blocked programs first; descend to warp and instruction details only when generated code or counters require it.

**Failure analysis:** Treating each lane of a blocked tensor as an independently scheduled CUDA thread leads to wrong assumptions about synchronization and storage.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
